<h1> Primary data collection for oil production in Texas <h1>

Note: Do not run this again unless newer information is required.<br>

All necessary data for texas oil procuction is obtained from https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/. We will be using the Production Data Query Dump dataset. This program extracts the necessary information from OG_LEASE_CYCLE_DATA_TABLE.dsv, which has oil and gas production data for every leases from 1993 to 2016 and from OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv, which has information about how the oil is dispensed out of the production plant. This program ignores gas wells and only focusses on the oil wells. The gas data available here is from the Casinghead gas, which is the natural gas that is found dissolved in crude oil and is produced alongside it (often flared).

The input of this program is a zip file which have the data set (texas_pdq.zip) from the production dump query. The output of this program saves two files og_lease_cycle.parquet and og_lease_cycle_disp.parquet in the folder pdq_lease_output in data. The users manual for the dataset obtained from the website is also saved alaongside this in the data/raw/texas folder.

Texas data files do not track information at the level of oil wells (at least the ones available for public). So all the production information is at the level of leases which may contain more than one well.

In [1]:
# ── CELL 1: Imports + Config ─────────────────────────────────────────────────
# Run this first after any kernel crash. Everything needed is in this one cell.

import zipfile
import logging
import warnings
import sys
import gc
from io import TextIOWrapper, BytesIO
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

OUTER_ZIP    = "../../../data/raw/texas/texas_pdq.zip"     # ← path to your outer zip
INNER_ZIP    = "PDQ_DSV.zip"                               # name of zip inside outer zip
OUT_DIR      = "../../../data/raw/texas/pdq_lease_output"  # where to save outputs
FORMAT       = "parquet"                                   # "parquet" or "csv"
CHUNKSIZE    = 150_000                                     # reduced for 16GB RAM
OIL_GAS_FILTER = "O"                                       # "O" = oil leases only
                                                           # "G" = gas leases only
                                                           # None = load everything
DO_MERGE     = True                                        # True = produce joined table

This is to select the relevant columns of both the dsv files (based on the information from the users manual) and to normalise the naming

In [2]:
# ── CELL 2: Constants ─────────────────────────────────────────────────────────

DELIMITER  = "}"
ENCODING   = "latin-1"
CYCLE_FILE = "OG_LEASE_CYCLE_DATA_TABLE.dsv"
DISP_FILE  = "OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv"

CYCLE_KEEP = [
    "OIL_GAS_CODE",
    "DISTRICT_NO",
    "LEASE_NO",           
    "CYCLE_YEAR_MONTH",
    "FIELD_NO",
    "LEASE_OIL_PROD_VOL",
    "LEASE_CSGD_PROD_VOL",
    "LEASE_CSGD_TOT_DISP",
]

OIL_DISP_LABELS = {
    "LEASE_OIL_DISPCD00_VOL": "oil_pipeline_bbl",
    "LEASE_OIL_DISPCD01_VOL": "oil_truck_bbl",
    "LEASE_OIL_DISPCD02_VOL": "oil_tankcar_bbl",
    "LEASE_OIL_DISPCD03_VOL": "oil_tank_cleaning_bbl",
    "LEASE_OIL_DISPCD04_VOL": "oil_circulating_bbl",
    "LEASE_OIL_DISPCD05_VOL": "oil_lost_stolen_bbl",
    "LEASE_OIL_DISPCD06_VOL": "oil_bsw_repressure_bbl",
    "LEASE_OIL_DISPCD07_VOL": "oil_legacy_bbl",
    "LEASE_OIL_DISPCD08_VOL": "oil_skimmed_bbl",
    "LEASE_OIL_DISPCD09_VOL": "oil_scrubber_bbl",
    "LEASE_OIL_DISPCD99_VOL": "oil_no_disp_code_bbl",
}
GAS_DISP_LABELS = {
    "LEASE_GAS_DISPCD01_VOL": "gas_field_ops_fuel_mcf",
    "LEASE_GAS_DISPCD02_VOL": "gas_transmission_mcf",
    "LEASE_GAS_DISPCD03_VOL": "gas_processing_plant_mcf",
    "LEASE_GAS_DISPCD04_VOL": "gas_vented_flared_mcf",
    "LEASE_GAS_DISPCD05_VOL": "gas_lift_mcf",
    "LEASE_GAS_DISPCD06_VOL": "gas_repressure_mcf",
    "LEASE_GAS_DISPCD07_VOL": "gas_carbon_black_mcf",
    "LEASE_GAS_DISPCD08_VOL": "gas_underground_storage_mcf",
    "LEASE_GAS_DISPCD09_VOL": "gas_separation_loss_mcf",
    "LEASE_GAS_DISPCD99_VOL": "gas_no_disp_code_mcf",
}
COND_DISP_LABELS = {
    "LEASE_COND_DISPCD00_VOL": "cond_pipeline_bbl",
    "LEASE_COND_DISPCD01_VOL": "cond_truck_bbl",
    "LEASE_COND_DISPCD02_VOL": "cond_tankcar_bbl",
    "LEASE_COND_DISPCD03_VOL": "cond_tank_cleaning_bbl",
    "LEASE_COND_DISPCD04_VOL": "cond_circulating_bbl",
    "LEASE_COND_DISPCD05_VOL": "cond_lost_stolen_bbl",
    "LEASE_COND_DISPCD06_VOL": "cond_bsw_repressure_bbl",
    "LEASE_COND_DISPCD07_VOL": "cond_legacy_bbl",
    "LEASE_COND_DISPCD08_VOL": "cond_skimmed_bbl",
    "LEASE_COND_DISPCD99_VOL": "cond_no_disp_code_bbl",
}
CSGD_DISP_LABELS = {
    "LEASE_CSGD_DISPCDE01_VOL": "csgd_field_ops_fuel_mcf",
    "LEASE_CSGD_DISPCDE02_VOL": "csgd_transmission_mcf",
    "LEASE_CSGD_DISPCDE03_VOL": "csgd_processing_plant_mcf",
    "LEASE_CSGD_DISPCDE04_VOL": "csgd_vented_flared_mcf",
    "LEASE_CSGD_DISPCDE05_VOL": "csgd_gas_lift_mcf",
    "LEASE_CSGD_DISPCDE06_VOL": "csgd_repressure_mcf",
    "LEASE_CSGD_DISPCDE07_VOL": "csgd_carbon_black_mcf",
    "LEASE_CSGD_DISPCDE08_VOL": "csgd_underground_storage_mcf",
    "LEASE_CSGD_DISPCDE99_VOL": "csgd_no_disp_code_mcf",
}
ALL_DISP_LABELS = {
    **OIL_DISP_LABELS, **GAS_DISP_LABELS,
    **COND_DISP_LABELS, **CSGD_DISP_LABELS,
}

DISP_KEYS = ["OIL_GAS_CODE", "DISTRICT_NO", "CYCLE_YEAR_MONTH", "FIELD_NO"]
DISP_KEEP = DISP_KEYS + list(ALL_DISP_LABELS.keys())
JOIN_KEYS = ["OIL_GAS_CODE", "DISTRICT_NO", "CYCLE_YEAR_MONTH", "FIELD_NO"]

log.info("✓ Cell 2 done — constants loaded.")


20:27:36 [INFO] ✓ Cell 2 done — constants loaded.


In [3]:
# ── CELL 3: Cleaning Functions ────────────────────────────────────────────────

def _strip_cols(df):
    df.columns = df.columns.str.strip()
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())
    return df

def _apply_filter(df):
    if OIL_GAS_FILTER and "OIL_GAS_CODE" in df.columns:
        df = df[df["OIL_GAS_CODE"].str.strip() == OIL_GAS_FILTER]
    return df

def _cast_cycle(df):
    num_cols = [c for c in df.columns if any(c.startswith(p) for p in
                ("LEASE_OIL_", "LEASE_GAS_", "LEASE_COND_", "LEASE_CSGD_"))]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    ym = df["CYCLE_YEAR_MONTH"].astype(str).str.zfill(6)
    df["PROD_DATE"] = pd.to_datetime(
        ym.str[:4] + "-" + ym.str[4:] + "-01", errors="coerce")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO", "FIELD_TYPE", "PROD_REPORT_FILED_FLAG"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _cast_disp(df):
    for col in [c for c in df.columns if "_DISPCD" in c or "_DISPCDE" in c]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["CYCLE_YEAR_MONTH"] = pd.to_numeric(
        df["CYCLE_YEAR_MONTH"], errors="coerce").astype("Int32")
    for col in ("OIL_GAS_CODE", "DISTRICT_NO"):
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

def _add_derived_disp_cols(df):
    present = set(df.columns)
    sold = ["LEASE_OIL_DISPCD00_VOL", "LEASE_OIL_DISPCD01_VOL", "LEASE_OIL_DISPCD02_VOL"]
    if all(c in present for c in sold):
        df["oil_sold_total_bbl"] = df[sold].sum(axis=1, min_count=1)
    g_flare, cg_flare = "LEASE_GAS_DISPCD04_VOL", "LEASE_CSGD_DISPCDE04_VOL"
    if g_flare in present and cg_flare in present:
        df["total_vented_flared_mcf"] = df[[g_flare, cg_flare]].sum(axis=1, min_count=1)
    elif g_flare in present:
        df["total_vented_flared_mcf"] = df[g_flare]
    elif cg_flare in present:
        df["total_vented_flared_mcf"] = df[cg_flare]
    g_proc, cg_proc = "LEASE_GAS_DISPCD03_VOL", "LEASE_CSGD_DISPCDE03_VOL"
    if g_proc in present and cg_proc in present:
        df["total_gas_to_processing_mcf"] = df[[g_proc, cg_proc]].sum(axis=1, min_count=1)
    return df

def clean_cycle_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in CYCLE_KEEP if c in chunk.columns]]
    chunk = _cast_cycle(chunk)
    vol_cols = [c for c in chunk.columns if c.endswith("_PROD_VOL")]
    chunk[vol_cols] = chunk[vol_cols].replace(0, pd.NA)
    return chunk

def clean_disp_chunk(chunk):
    chunk = _strip_cols(chunk)
    chunk = _apply_filter(chunk)
    if chunk.empty:
        return chunk
    chunk = chunk[[c for c in DISP_KEEP if c in chunk.columns]]
    chunk = _cast_disp(chunk)
    chunk = _add_derived_disp_cols(chunk)
    chunk = chunk.rename(columns={
        k: v for k, v in ALL_DISP_LABELS.items() if k in chunk.columns})
    return chunk

log.info("✓ Cell 3 done — cleaning functions defined.")


20:27:36 [INFO] ✓ Cell 3 done — cleaning functions defined.


In [4]:
# ── CELL 4: Reader Functions ──────────────────────────────────────────────────

def open_inner_zip(outer_path, inner_name):
    with zipfile.ZipFile(outer_path, "r") as outer:
        outer_contents = outer.namelist()
        log.info("Files in outer zip: %s", outer_contents)
        match = next((f for f in outer_contents
                      if f.upper() == inner_name.upper()), None)
        if match is None:
            raise FileNotFoundError(
                f"'{inner_name}' not found in outer zip.\nAvailable: {outer_contents}")
        log.info("Opening inner zip: %s", match)
        inner_bytes = BytesIO(outer.read(match))
    return zipfile.ZipFile(inner_bytes, "r")

def read_chunked(inner_zf, filename, cleaner):
    try:
        inner_zf.getinfo(filename)
    except KeyError:
        raise FileNotFoundError(
            f"'{filename}' not found.\nAvailable: {inner_zf.namelist()}")
    log.info("Reading %s (chunk size = %s) ...", filename, f"{CHUNKSIZE:,}")
    chunks, total_in, total_out = [], 0, 0
    with inner_zf.open(filename) as raw:
        reader = pd.read_csv(
            TextIOWrapper(raw, encoding=ENCODING),
            sep=DELIMITER,
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="warn",
        )
        for i, chunk in enumerate(reader, 1):
            total_in += len(chunk)
            cleaned = cleaner(chunk)
            if not cleaned.empty:
                chunks.append(cleaned)
                total_out += len(cleaned)
            del chunk, cleaned
            gc.collect()
            log.info("  chunk %3d — kept %s / %s rows",
                     i, f"{total_out:,}", f"{total_in:,}")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    log.info("✓ Done: %s rows x %s columns", f"{len(df):,}", len(df.columns))
    return df

log.info("✓ Cell 4 done — reader functions defined.")


20:27:36 [INFO] ✓ Cell 4 done — reader functions defined.


In [5]:
# ── CELL 5: Load OG_LEASE_CYCLE ───────────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_cycle = read_chunked(inner_zf, CYCLE_FILE, clean_cycle_chunk)
inner_zf.close()
gc.collect()

print("\nShape      :", df_cycle.shape)
print("Date range :", df_cycle["PROD_DATE"].min(), "→", df_cycle["PROD_DATE"].max())
print("Memory     :", f"{df_cycle.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns    :", df_cycle.columns.tolist())
df_cycle.head(3)


20:27:36 [INFO] 
── Loading OG_LEASE_CYCLE ──
20:27:36 [INFO] Files in outer zip: ['PDQ_DSV.zip']
20:27:36 [INFO] Opening inner zip: PDQ_DSV.zip
20:27:42 [INFO] Reading OG_LEASE_CYCLE_DATA_TABLE.dsv (chunk size = 150,000) ...
20:27:42 [INFO]   chunk   1 — kept 44,754 / 150,000 rows
20:27:42 [INFO]   chunk   2 — kept 99,294 / 300,000 rows
20:27:43 [INFO]   chunk   3 — kept 156,248 / 450,000 rows
20:27:43 [INFO]   chunk   4 — kept 225,294 / 600,000 rows
20:27:44 [INFO]   chunk   5 — kept 280,138 / 750,000 rows
20:27:44 [INFO]   chunk   6 — kept 368,123 / 900,000 rows
20:27:44 [INFO]   chunk   7 — kept 423,924 / 1,050,000 rows
20:27:45 [INFO]   chunk   8 — kept 508,491 / 1,200,000 rows
20:27:45 [INFO]   chunk   9 — kept 561,626 / 1,350,000 rows
20:27:45 [INFO]   chunk  10 — kept 596,273 / 1,500,000 rows
20:27:46 [INFO]   chunk  11 — kept 636,592 / 1,650,000 rows
20:27:46 [INFO]   chunk  12 — kept 705,075 / 1,800,000 rows
20:27:46 [INFO]   chunk  13 — kept 759,134 / 1,950,000 rows
20:27:47

,OIL_GAS_CODE,DISTRICT_NO,LEASE_NO,CYCLE_YEAR_MONTH,FIELD_NO,LEASE_OIL_PROD_VOL,LEASE_CSGD_PROD_VOL,LEASE_CSGD_TOT_DISP,PROD_DATE
0,O,02,05905,202601,57911234,<NA>,<NA>,0,2026-01-01
1,O,02,05905,202602,57911234,<NA>,<NA>,0,2026-02-01
2,O,02,05905,202603,57911234,<NA>,<NA>,0,2026-03-01


In [6]:
# ── CELL 6: Load OG_LEASE_CYCLE_DISP ─────────────────────────────────────────

log.info("\n── Loading OG_LEASE_CYCLE_DISP ──")
inner_zf = open_inner_zip(OUTER_ZIP, INNER_ZIP)
df_disp = read_chunked(inner_zf, DISP_FILE, clean_disp_chunk)
inner_zf.close()
gc.collect()

print("\nShape  :", df_disp.shape)
print("Memory :", f"{df_disp.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print("\nColumns:", df_disp.columns.tolist())

dupes = df_disp.columns[df_disp.columns.duplicated()].tolist()
if dupes:
    log.error("Duplicate columns in df_disp: %s", dupes)
else:
    log.info("✓ No duplicate columns.")

df_disp.head(3)


20:30:56 [INFO] 
── Loading OG_LEASE_CYCLE_DISP ──
20:30:56 [INFO] Files in outer zip: ['PDQ_DSV.zip']
20:30:56 [INFO] Opening inner zip: PDQ_DSV.zip
20:31:02 [INFO] Reading OG_LEASE_CYCLE_DISP_DATA_TABLE.dsv (chunk size = 150,000) ...
20:31:03 [INFO]   chunk   1 — kept 47,657 / 150,000 rows
20:31:04 [INFO]   chunk   2 — kept 108,852 / 300,000 rows
20:31:05 [INFO]   chunk   3 — kept 148,948 / 450,000 rows
20:31:05 [INFO]   chunk   4 — kept 195,204 / 600,000 rows
20:31:06 [INFO]   chunk   5 — kept 233,180 / 750,000 rows
20:31:07 [INFO]   chunk   6 — kept 283,864 / 900,000 rows
20:31:08 [INFO]   chunk   7 — kept 346,512 / 1,050,000 rows
20:31:09 [INFO]   chunk   8 — kept 392,278 / 1,200,000 rows
20:31:10 [INFO]   chunk   9 — kept 481,437 / 1,350,000 rows
20:31:11 [INFO]   chunk  10 — kept 525,218 / 1,500,000 rows
20:31:12 [INFO]   chunk  11 — kept 591,342 / 1,650,000 rows
20:31:13 [INFO]   chunk  12 — kept 666,076 / 1,800,000 rows
20:31:13 [INFO]   chunk  13 — kept 710,671 / 1,950,000 ro

,OIL_GAS_CODE,DISTRICT_NO,CYCLE_YEAR_MONTH,FIELD_NO,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,oil_circulating_bbl,oil_lost_stolen_bbl,...,csgd_processing_plant_mcf,csgd_vented_flared_mcf,csgd_gas_lift_mcf,csgd_repressure_mcf,csgd_carbon_black_mcf,csgd_underground_storage_mcf,csgd_no_disp_code_mcf,oil_sold_total_bbl,total_vented_flared_mcf,total_gas_to_processing_mcf
0,O,08,202603,89812001,0,165,0,0,0,0,...,0,0,0,0,0,0,0,165,0,0
1,O,10,202601,19541001,0,146,0,0,0,0,...,0,0,0,0,0,0,0,146,0,0
2,O,10,202602,19541001,0,154,0,0,0,0,...,0,0,0,0,0,0,0,154,0,0


In [7]:
# ── CELL 7: Save separately — no merge ────────────────────────────────────────

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)

def save_df(df, name):
    path = out / f"{name}.{FORMAT}"
    if FORMAT == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    log.info("Saved %s  (%.1f MB)", path.name, path.stat().st_size / 1_048_576)

log.info("Saving df_cycle ...")
save_df(df_cycle, "og_lease_cycle")
del df_cycle
gc.collect()
log.info("✓ df_cycle saved and freed from memory.")

log.info("Saving df_disp ...")
save_df(df_disp, "og_lease_cycle_disp")
del df_disp
gc.collect()
log.info("✓ df_disp saved and freed from memory.")

log.info("\n✓ All done. Files saved to: %s", out.resolve())
for f in sorted(out.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1_048_576:.1f} MB)")

20:35:35 [INFO] Saving df_cycle ...
20:35:40 [INFO] Saved og_lease_cycle.parquet  (139.4 MB)
20:35:40 [INFO] ✓ df_cycle saved and freed from memory.
20:35:40 [INFO] Saving df_disp ...
20:35:46 [INFO] Saved og_lease_cycle_disp.parquet  (139.2 MB)
20:35:46 [INFO] ✓ df_disp saved and freed from memory.
20:35:46 [INFO] 
✓ All done. Files saved to: /Users/sabare/Desktop/Projects/summer26-permian-flaring/data/raw/texas/pdq_lease_output
  og_lease_cycle.parquet  (139.4 MB)
  og_lease_cycle_disp.parquet  (139.2 MB)
